# Setup

In [5]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

BASE_PATH = "/app/.data"

import sys
print(sys.executable)

d:\Data Engineering\Assignment\saas-analytics-pipeline\.venv\Scripts\python.exe


## Dataset

In [10]:
import pandas as pd 
import numpy as np
import random
from datetime import datetime, timedelta

# =========================
# SEED (REPRODUCIBILITY)
# =========================
# Ensure consistent dataset across runs for testing & portfolio purposes
np.random.seed(42)
random.seed(42)

# =========================
# CONFIGURATION (Business Scale Setup)
# =========================
N_ACCOUNTS = 10000             # Total unique accounts
N_SUBSCRIPTIONS_TARGET = 100000  # Approximate total subscription events
N_USAGE = 500000                # Number of feature usage records
N_TICKETS = 50000               # Number of support tickets

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2026, 12, 31)

industries = ["Technology", "Finance", "E-commerce & Retail", "Healthcare & Life Sciences",
              "Education", "Energy & Resources", "Industrial & Logistics",
              "Consumer & Lifestyle", "Professional Services"]

countries = ["Singapore", "Brunei", "Cambodia", "Indonesia", "Laos",
             "Malaysia", "Myanmar", "Philippines"]

referrals = ["organic", "ads", "partner", "outbound", "event", "other"]
plans = ["Free", "Basic", "Pro", "Enterprise"]
features = ["dashboard", "api", "export", "automation", "integration"]
priorities = ["low", "medium", "high"]

# =========================
# HELPER FUNCTIONS
# =========================
def random_date(start, end):
    """Return a random datetime between start and end"""
    if end <= start:
        return start
    delta = end - start
    total_seconds = int(delta.total_seconds())
    random_second = random.randint(0, total_seconds)
    return start + timedelta(seconds=random_second)

def plan_price(plan, seats):
    """
    Return revenue for a given plan & number of seats.
    Free plan always 0, Enterprise has variable pricing.
    """
    if plan == "Basic":
        return seats * 10
    elif plan == "Pro":
        return seats * 25
    elif plan == "Enterprise":
        return seats * random.randint(40, 80)
    return 0

# =========================
# 1. ACCOUNT TABLE
# =========================
# Simulate SaaS customers with realistic signup behavior
accounts = []

for i in range(N_ACCOUNTS):
    account_id = f"A{i:05d}"
    signup_date = random_date(START_DATE, END_DATE)
    plan = random.choices(plans, weights=[0.3,0.3,0.25,0.15])[0]  # Funnel: most start Free/Basic
    seats = random.randint(50, 500) if plan == "Enterprise" else random.randint(1,50)
    is_trial = plan == "Free" or random.random() < 0.4

    accounts.append([
        account_id,
        f"Company_{i}",
        random.choice(industries),
        random.choice(countries),
        signup_date,
        random.choice(referrals),
        plan,
        seats,
        is_trial,
        False  # churn_flag updated later
    ])

account_df = pd.DataFrame(accounts, columns=[
    "account_id","account_name","industry","country",
    "signup_date","referral_source","plan_tier","seats",
    "is_trial","churn_flag"
])

# =========================
# 2. SUBSCRIPTION TABLE
# =========================
# Tracks all subscription events, upgrades/downgrades, churn, MRR/ARR
subs = []
sub_id_counter = 0

for _, acc in account_df.iterrows():
    if sub_id_counter >= N_SUBSCRIPTIONS_TARGET:
        break

    account_id = acc.account_id
    current_date = acc.signup_date
    last_plan = acc.plan_tier
    last_seats = acc.seats
    n_events = random.randint(5, 15)
    cycles = 0
    max_cycles = 20
    start_idx = len(subs)
    seen_plans = set()
    is_first_subscription = True

    while cycles < max_cycles:
        if current_date >= END_DATE:
            break
        cycles += 1
        sub_id = f"S{sub_id_counter:06d}"
        sub_id_counter += 1

        # Duration of this subscription period (30-180 days)
        duration_days = random.randint(30, 180)
        end_date = min(current_date + timedelta(days=duration_days), END_DATE)

        # Plan change simulation
        move = random.choice(["same","upgrade","downgrade"])
        if move == "upgrade":
            plan_idx = min(plans.index(last_plan)+1, len(plans)-1)
            plan = plans[plan_idx]
            seats = last_seats + random.randint(1,20)
        elif move == "downgrade":
            plan_idx = max(plans.index(last_plan)-1,0)
            plan = plans[plan_idx]
            seats = max(1,last_seats - random.randint(1,10))
        else:
            plan = last_plan
            seats = last_seats

        # Revenue calculation
        revenue = float(round(plan_price(plan, seats),2))
        price = round(revenue / seats,2)
            # Tentukan rate dasar per plan
        rates = {
            "Basic": 0.05,
            "Pro": 0.10,
            "Enterprise": 0.20,
            "Free": 0
        }
            # Logika: Hanya 10% data yang dapet diskon
        if random.random() < 0.10:
            discount_rate = rates[plan]
        else:
            discount_rate = 0.0

        # Hitung nilai diskon & net revenue
        discount_value = round(revenue * discount_rate, 2)
        discount_rate = {"Basic":0.05,"Pro":0.10,"Enterprise":0.20,"Free":0}[plan]
        discount_value = round(revenue * discount_rate,2)
        net_revenue = round(revenue - discount_value,2)
        mrr = net_revenue
        arr = round(mrr*12,2)

        # Trial logic: first subscription or first time using this plan
        is_trial = is_first_subscription or (plan not in seen_plans)

        # Upgrade / Downgrade flags
        if is_first_subscription:
            upgrade_flag = False
            downgrade_flag = False
        else:
            upgrade_flag = move=="upgrade"
            downgrade_flag = move=="downgrade"

        churn = random.random() < 0.1

        subs.append([
            sub_id,
            account_id,
            current_date,
            end_date,
            plan,
            seats,
            price,
            discount_value,
            mrr,
            arr,
            is_trial,
            upgrade_flag,
            downgrade_flag,
            churn,
            random.choice(["monthly","annual"]),
            False if plan=="Free" else random.random()<0.85
        ])

        seen_plans.add(plan)
        is_first_subscription = False

        # Reactivation / churn logic
        if churn:
            if random.random() < 0.3:  # chance to reactivate later
                gap_days = random.randint(30,180)
                current_date = min(end_date + timedelta(days=gap_days), END_DATE)
                last_plan = random.choice(["Basic","Pro"])
                last_seats = random.randint(1,20)
                continue
            else:
                break

        current_date = end_date
        last_plan = plan
        last_seats = seats
        if len(subs)-start_idx >= n_events:
            break

subscription_df = pd.DataFrame(subs, columns=[
    "subscription_id","account_id","start_date","end_date",
    "plan_tier","seats","price","discount_value","mrr_amount","arr_amount",
    "is_trial","upgrade_flag",
    "downgrade_flag","churn_flag","billing_frequency","auto_renew_flag"
])

# Detect reactivation events
subscription_df = subscription_df.sort_values(["account_id","start_date"])
subscription_df["next_start"] = subscription_df.groupby("account_id")["start_date"].shift(-1)
subscription_df["is_reactivation"] = (
    (subscription_df["churn_flag"]==True) &
    (subscription_df["next_start"].notna())
)

# =========================
# 3. CHURN TABLE
# =========================
# Detailed churn events with reason & refund logic
churn_rows = []
for i,row in subscription_df[subscription_df["churn_flag"]].iterrows():
    churn_date = row["end_date"]
    refund = 0
    if pd.notna(churn_date):
        duration_days = (churn_date - row["start_date"]).days
        if duration_days < 30:
            duration_months = max(0.1,duration_days/20)
            actual_revenue = row["mrr_amount"]*duration_months
            refund = round(actual_revenue*0.5,2)
    churn_rows.append([
        f"C{i:06d}",
        row["account_id"],
        churn_date,
        random.choice(["price_too_high","missing_features","bugs","competitor","no_longer_needed"]),
        refund,
        row["upgrade_flag"],
        row["downgrade_flag"],
        row["is_reactivation"],
        random.choice([None,"Too expensive","Buggy","Not useful"])
    ])

churn_df = pd.DataFrame(churn_rows, columns=[
    "churn_event_id","account_id","churn_date","reason_code",
    "refund_amount_usd","preceding_upgrade_flag",
    "preceding_downgrade_flag","is_reactivation","feedback_text"
])

# Update account churn flag
account_df.loc[account_df.account_id.isin(churn_df.account_id),"churn_flag"] = True

# =========================
# 4. FEATURE USAGE TABLE
# =========================
# Logs actual usage of platform features per subscription
usage_rows = []
subs_sample = subscription_df.sample(N_USAGE, replace=True)

for i,row in subs_sample.iterrows():
    end_date = row["end_date"] if pd.notna(row["end_date"]) else END_DATE
    usage_date = random_date(row["start_date"], end_date)
    base_usage = {"Free":5,"Basic":20,"Pro":50,"Enterprise":100}[row["plan_tier"]]
    usage_count = max(0,int(np.random.normal(base_usage, base_usage*0.3)))
    duration = usage_count * random.randint(10,60)
    usage_rows.append([
        f"U{i:06d}",
        row["subscription_id"],
        usage_date,
        random.choice(features),
        usage_count,
        duration,
        random.randint(0,3),
        random.random()<0.1
    ])

usage_df = pd.DataFrame(usage_rows, columns=[
    "usage_id","subscription_id","usage_date",
    "feature_name","usage_count","usage_duration_secs",
    "error_count","is_beta_feature"
])

# =========================
# 5. SUPPORT TICKETS TABLE
# =========================
# Simulate customer support tickets with resolution & satisfaction metrics
tickets = []

for i in range(N_TICKETS):
    acc = account_df.sample(1).iloc[0]
    submitted = random_date(acc.signup_date, END_DATE)
    priority = random.choice(priorities)
    escalation = random.random() < (0.3 if priority=="high" else 0.1)
    resolution_time = {"low":round(random.uniform(24,72),1),
                       "medium":round(random.uniform(12,48),1),
                       "high":round(random.uniform(2,24),1)}[priority]
    satisfaction = None
    if random.random()>0.4:
        satisfaction = round(max(1,5-(resolution_time/24)-(2 if escalation else 0)),2)
    tickets.append([
        f"T{i:06d}",
        acc.account_id,
        submitted,
        submitted+timedelta(hours=resolution_time),
        resolution_time,
        priority,
        random.randint(5,120),
        satisfaction,
        escalation
    ])

ticket_df = pd.DataFrame(tickets, columns=[
    "ticket_id","account_id","submitted_at","closed_at",
    "resolution_time_hours","priority","first_response_time_minutes",
    "satisfaction_score","escalation_flag"
])


# =========================
# OUTPUT
# =========================
print("account_df:", account_df.shape)
print("subscription_df:", subscription_df.shape)
print("churn_df:", churn_df.shape)
print("usage_df:", usage_df.shape)
print("ticket_df:", ticket_df.shape)

account_df: (10000, 10)
subscription_df: (43351, 18)
churn_df: (4285, 9)
usage_df: (500000, 8)
ticket_df: (50000, 9)


In [12]:
accounts =  account_df.copy()
subscriptions =  subscription_df.copy()
churn = churn_df.copy()
usage = usage_df.copy()
tickets = ticket_df.copy()

In [13]:
subscriptions.info()

<class 'pandas.DataFrame'>
RangeIndex: 43351 entries, 0 to 43350
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   subscription_id    43351 non-null  str           
 1   account_id         43351 non-null  str           
 2   start_date         43351 non-null  datetime64[us]
 3   end_date           43351 non-null  datetime64[us]
 4   plan_tier          43351 non-null  str           
 5   seats              43351 non-null  int64         
 6   price              43351 non-null  float64       
 7   discount_value     43351 non-null  float64       
 8   mrr_amount         43351 non-null  float64       
 9   arr_amount         43351 non-null  float64       
 10  is_trial           43351 non-null  bool          
 11  upgrade_flag       43351 non-null  bool          
 12  downgrade_flag     43351 non-null  bool          
 13  churn_flag         43351 non-null  bool          
 14  billing_frequency

In [14]:
import pandas as pd

def profile_table(df, table_name="Table"):
    rows = []
    
    for col in df.columns:
        series = df[col].dropna()
        nunique = series.nunique()
        
        # Get unique values first
        unique_vals = series.unique()
        
        if nunique <= 5:
            values = list(unique_vals)
        else:
            # Sample from the Series or Array, then convert to list
            values = pd.Series(unique_vals).sample(n=min(4, nunique), random_state=42).tolist()
        
        rows.append({
            # "table": table_name,
            "column": col,
            "dtype": str(df[col].dtype),
            "n_unique": nunique,
            "sample_values": values
        })
    
    return pd.DataFrame(rows)


In [15]:
tables = {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "churn": churn,
    "usage": usage,
    "tickets": tickets
}

profile_results = pd.concat(
    [profile_table(df, name) for name, df in tables.items()],
    ignore_index=True
)

for name, df in tables.items():
    print(f"\n===== {name.upper()} =====")
    display(profile_table(df, name))


===== ACCOUNTS =====


,column,dtype,n_unique,sample_values
0,account_id,str,10000,"[A06252, A04684, A01731, A04742]"
1,account_name,str,10000,"[Company_6252, Company_4684, Company_1731, Com..."
2,industry,str,9,"[Industrial & Logistics, Professional Services..."
3,country,str,8,"[Brunei, Cambodia, Indonesia, Philippines]"
4,signup_date,datetime64[us],9999,"[2024-09-10 02:42:03, 2026-03-16 21:17:11, 202..."
5,referral_source,str,6,"[ads, event, outbound, partner]"
6,plan_tier,str,4,"[Free, Pro, Basic, Enterprise]"
7,seats,int64,490,"[211, 74, 119, 309]"
8,is_trial,bool,2,"[True, False]"
9,churn_flag,bool,2,"[False, True]"



===== SUBSCRIPTIONS =====


,column,dtype,n_unique,sample_values
0,subscription_id,str,43351,"[S036813, S008763, S017515, S024525]"
1,account_id,str,10000,"[A06252, A04684, A01731, A04742]"
2,start_date,datetime64[us],43337,"[2024-07-18 14:54:40, 2025-05-31 10:47:43, 202..."
3,end_date,datetime64[us],36992,"[2026-01-19 20:32:11, 2026-04-05 07:24:20, 202..."
4,plan_tier,str,4,"[Basic, Free, Pro, Enterprise]"
5,seats,int64,533,"[9, 288, 95, 359]"
6,price,float64,44,"[77.0, 46.0, 41.0, 44.0]"
7,discount_value,float64,4625,"[5846.4, 648.6, 3126.2, 2524.8]"
8,mrr_amount,float64,4838,"[369.6, 2160.8, 649.6, 4620.0]"
9,arr_amount,float64,4838,"[4435.2, 25929.6, 7795.2, 55440.0]"



===== CHURN =====


,column,dtype,n_unique,sample_values
0,churn_event_id,str,4285,"[C038008, C028575, C030841, C027490]"
1,account_id,str,4032,"[A02786, A01625, A01655, A06897]"
2,churn_date,datetime64[us],3610,"[2025-11-30 15:17:01, 2026-08-14 11:42:43, 202..."
3,reason_code,str,5,"[missing_features, bugs, no_longer_needed, pri..."
4,refund_amount_usd,float64,129,"[507.38, 2097.6, 171.2, 194.06]"
5,preceding_upgrade_flag,bool,2,"[True, False]"
6,preceding_downgrade_flag,bool,2,"[False, True]"
7,is_reactivation,bool,2,"[False, True]"
8,feedback_text,str,3,"[Buggy, Not useful, Too expensive]"



===== USAGE =====


,column,dtype,n_unique,sample_values
0,usage_id,str,43351,"[U006019, U032140, U024193, U030360]"
1,subscription_id,str,43351,"[S006019, S032140, S024193, S030360]"
2,usage_date,datetime64[us],498127,"[2024-10-30 03:56:40, 2026-02-22 13:33:47, 202..."
3,feature_name,str,5,"[automation, api, dashboard, integration, export]"
4,usage_count,int64,215,"[190, 213, 133, 176]"
5,usage_duration_secs,int64,4006,"[522, 1284, 3465, 1310]"
6,error_count,int64,4,"[0, 2, 1, 3]"
7,is_beta_feature,bool,2,"[False, True]"



===== TICKETS =====


,column,dtype,n_unique,sample_values
0,ticket_id,str,50000,"[T033553, T009427, T000199, T012447]"
1,account_id,str,9944,"[A05745, A03350, A08497, A08267]"
2,submitted_at,datetime64[us],49967,"[2025-12-14 11:32:58, 2026-12-24 20:10:50, 202..."
3,closed_at,datetime64[us],49977,"[2026-04-29 14:01:14, 2026-10-01 12:03:15, 202..."
4,resolution_time_hours,float64,701,"[70.4, 16.1, 14.4, 65.3]"
5,priority,str,3,"[medium, low, high]"
6,first_response_time_minutes,int64,116,"[33, 49, 9, 65]"
7,satisfaction_score,float64,393,"[3.04, 3.16, 2.72, 3.33]"
8,escalation_flag,bool,2,"[False, True]"


### Accounts

In [ ]:
accounts.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   account_id       10000 non-null  str           
 1   account_name     10000 non-null  str           
 2   industry         10000 non-null  str           
 3   country          10000 non-null  str           
 4   signup_date      10000 non-null  datetime64[us]
 5   referral_source  10000 non-null  str           
 6   plan_tier        10000 non-null  str           
 7   seats            10000 non-null  int64         
 8   is_trial         10000 non-null  bool          
 9   churn_flag       10000 non-null  bool          
dtypes: bool(2), datetime64[us](1), int64(1), str(6)
memory usage: 644.7 KB


In [ ]:
accounts.sample()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
438,A00438,Company_438,Industrial & Logistics,Cambodia,2024-01-13 12:35:39,outbound,Pro,8,False,False


In [ ]:
set(accounts['industry'].to_list())

{'Consumer & Lifestyle',
 'E-commerce & Retail',
 'Education',
 'Energy & Resources',
 'Finance',
 'Healthcare & Life Sciences',
 'Industrial & Logistics',
 'Professional Services',
 'Technology'}

In [ ]:
set(accounts['plan_tier'].to_list())

{'Basic', 'Enterprise', 'Free', 'Pro'}

In [ ]:
print(f"signup data from date {accounts['signup_date'].min()} until {accounts['signup_date'].max()}")

signup data from date 2024-01-01 00:21:12 until 2026-02-27 23:36:22


In [ ]:
set(accounts['is_trial'].to_list())

{False, True}

In [ ]:
set(accounts['churn_flag'].to_list())

{False, True}

### Subscriptions

In [ ]:
subscriptions.info()

<class 'pandas.DataFrame'>
RangeIndex: 35820 entries, 0 to 35819
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   subscription_id    35820 non-null  str           
 1   account_id         35820 non-null  str           
 2   start_date         35820 non-null  datetime64[us]
 3   end_date           33030 non-null  datetime64[us]
 4   plan_tier          35820 non-null  str           
 5   seats              35820 non-null  int64         
 6   price              35820 non-null  float64       
 7   revenue            35820 non-null  float64       
 8   mrr_amount         35820 non-null  float64       
 9   arr_amount         35820 non-null  float64       
 10  discount_value     35820 non-null  float64       
 11  net_revenue        35820 non-null  float64       
 12  is_trial           35820 non-null  bool          
 13  upgrade_flag       35820 non-null  bool          
 14  downgrade_flag   

In [ ]:
subscriptions.sample()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,price,revenue,mrr_amount,arr_amount,discount_value,net_revenue,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,next_start,is_reactivation
35242,S035242,A09838,2025-11-18 16:53:56,2026-02-28,Pro,42,25.0,1050.0,945.0,11340.0,105.0,945.0,True,False,False,False,annual,True,NaT,False


In [ ]:
subscriptions[subscriptions['churn_flag']==True]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,price,revenue,mrr_amount,arr_amount,discount_value,net_revenue,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,next_start,is_reactivation
5,S000005,A00001,2024-06-08 05:17:49,2024-08-05 05:17:49,Pro,35,25.0,875.0,787.5,9450.0,87.5,787.5,True,False,False,True,annual,True,NaT,False
8,S000008,A00002,2024-11-05 13:12:42,2024-12-21 13:12:42,Free,35,0.0,0.0,0.0,0.0,0.0,0.0,False,False,True,True,monthly,False,2025-03-30 13:12:42,True
14,S000014,A00003,2025-12-22 16:49:57,NaT,Free,34,0.0,0.0,0.0,0.0,0.0,0.0,False,False,True,True,annual,False,NaT,False
15,S000015,A00004,2025-10-18 12:12:24,NaT,Free,1,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,True,monthly,False,NaT,False
18,S000018,A00005,2025-12-12 04:42:59,NaT,Free,23,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True,monthly,False,NaT,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35767,S035767,A09985,2026-01-17 02:24:31,2026-02-24 02:24:31,Basic,37,10.0,370.0,351.5,4218.0,18.5,351.5,True,True,False,True,annual,True,NaT,False
35776,S035776,A09987,2025-09-14 01:57:21,NaT,Free,29,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True,annual,False,NaT,False
35791,S035791,A09990,2026-02-03 02:57:22,2026-02-28 00:00:00,Enterprise,271,71.0,19241.0,15392.8,184713.6,3848.2,15392.8,False,True,False,True,annual,True,NaT,False
35792,S035792,A09991,2025-01-20 11:56:07,2025-05-21 11:56:07,Pro,84,25.0,2100.0,1890.0,22680.0,210.0,1890.0,True,False,False,True,annual,True,NaT,False


In [ ]:
churn[churn['is_reactivation'] == True]

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
1,C000008,A00002,2024-12-21 13:12:42,competitor,0.0,False,True,True,Buggy
9,C000073,A00022,2025-07-04 13:24:49,no_longer_needed,0.0,False,False,True,Not useful
15,C000112,A00033,2025-02-20 14:59:44,missing_features,0.0,False,False,True,NaN
18,C000167,A00045,2025-11-26 02:58:07,bugs,0.0,False,False,True,Buggy
34,C000361,A00102,2025-10-22 07:55:32,price_too_high,0.0,True,False,True,Not useful
...,...,...,...,...,...,...,...,...,...
3502,C034941,A09763,2024-04-01 15:42:10,price_too_high,0.0,False,False,True,NaN
3506,C034989,A09774,2025-07-11 04:38:14,missing_features,0.0,True,False,True,NaN
3510,C035021,A09784,2025-10-16 01:58:37,competitor,0.0,False,False,True,Buggy
3534,C035420,A09892,2025-11-28 10:52:06,price_too_high,0.0,False,True,True,Too expensive


In [ ]:
churn[churn['refund_amount_usd'] > 0]

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
16,C000152,A00041,2026-02-28,bugs,1044.00,True,False,False,NaN
32,C000347,A00097,2026-02-28,competitor,54.47,False,True,False,Not useful
33,C000349,A00099,2026-02-28,missing_features,3061.52,False,False,False,Not useful
74,C000724,A00203,2026-02-28,no_longer_needed,2088.10,False,True,False,Not useful
123,C001163,A00341,2026-02-28,bugs,376.68,True,False,False,NaN
...,...,...,...,...,...,...,...,...,...
3473,C034664,A09680,2026-02-28,competitor,429.88,False,True,False,NaN
3511,C035022,A09784,2026-02-28,missing_features,72.00,True,False,False,Too expensive
3514,C035076,A09796,2026-02-28,price_too_high,292.64,False,False,False,Not useful
3516,C035122,A09808,2026-02-28,price_too_high,87.40,False,False,False,NaN


In [ ]:
subscriptions[subscriptions['account_id'] == 'A00179']

,subscription_id,account_id,start_date,end_date,plan_tier,seats,price,revenue,mrr_amount,arr_amount,discount_value,net_revenue,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,next_start,is_reactivation
632,S000632,A00179,2024-12-10 03:58:24,2025-02-13 03:58:24,Enterprise,72,80.0,5760.0,4608.0,55296.0,1152.0,4608.0,True,False,False,False,monthly,True,2025-02-13 03:58:24,False
633,S000633,A00179,2025-02-13 03:58:24,2025-07-09 03:58:24,Enterprise,72,74.0,5328.0,4262.4,51148.8,1065.6,4262.4,False,False,False,False,monthly,True,2025-07-09 03:58:24,False
634,S000634,A00179,2025-07-09 03:58:24,2025-08-18 03:58:24,Enterprise,90,65.0,5850.0,4680.0,56160.0,1170.0,4680.0,False,True,False,False,annual,True,2025-08-18 03:58:24,False
635,S000635,A00179,2025-08-18 03:58:24,2025-09-17 03:58:24,Enterprise,91,79.0,7189.0,5751.2,69014.4,1437.8,5751.2,False,True,False,False,annual,True,2025-09-17 03:58:24,False
636,S000636,A00179,2025-09-17 03:58:24,2026-02-28 00:00:00,Pro,84,25.0,2100.0,1890.0,22680.0,210.0,1890.0,True,False,True,False,annual,True,NaT,False


- data end_date banyak yang null, tambahkan logic based on billing frequency
- untuk 'is_trial' mr_amount & arr_amount adalah 0
- mrr_amount 

In [ ]:
churn[churn['refund_amount_usd'] > 0]

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
16,C000152,A00041,2026-02-28,bugs,1044.00,True,False,False,NaN
32,C000347,A00097,2026-02-28,competitor,54.47,False,True,False,Not useful
33,C000349,A00099,2026-02-28,missing_features,3061.52,False,False,False,Not useful
74,C000724,A00203,2026-02-28,no_longer_needed,2088.10,False,True,False,Not useful
123,C001163,A00341,2026-02-28,bugs,376.68,True,False,False,NaN
...,...,...,...,...,...,...,...,...,...
3473,C034664,A09680,2026-02-28,competitor,429.88,False,True,False,NaN
3511,C035022,A09784,2026-02-28,missing_features,72.00,True,False,False,Too expensive
3514,C035076,A09796,2026-02-28,price_too_high,292.64,False,False,False,Not useful
3516,C035122,A09808,2026-02-28,price_too_high,87.40,False,False,False,NaN


In [ ]:
churn[churn['account_id'] == 'A0001']

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text


In [ ]:
subscriptions[(subscriptions['churn_flag'] == True)].sort_values('start_date')

,subscription_id,account_id,start_date,end_date,plan_tier,seats,price,revenue,mrr_amount,arr_amount,discount_value,net_revenue,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,next_start,is_reactivation
9463,S009463,A02619,2024-01-02 00:24:22,2024-04-06 00:24:22,Basic,12,10.0,120.0,114.0,1368.0,6.0,114.0,True,False,False,True,annual,True,NaT,False
7224,S007224,A02010,2024-01-02 02:54:15,2024-02-06 02:54:15,Pro,43,25.0,1075.0,967.5,11610.0,107.5,967.5,True,False,False,True,monthly,True,NaT,False
1627,S001627,A00469,2024-01-03 06:30:31,2024-02-02 06:30:31,Enterprise,41,59.0,2419.0,1935.2,23222.4,483.8,1935.2,True,False,False,True,monthly,True,NaT,False
32192,S032192,A08978,2024-01-04 00:13:20,2024-05-08 00:13:20,Enterprise,66,75.0,4950.0,3960.0,47520.0,990.0,3960.0,True,False,False,True,monthly,True,NaT,False
1483,S001483,A00429,2024-01-04 20:12:11,2024-03-16 20:12:11,Enterprise,483,74.0,35742.0,28593.6,343123.2,7148.4,28593.6,True,False,False,True,annual,True,NaT,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2525,S002525,A00715,2026-02-27 12:09:04,2026-02-28 00:00:00,Basic,9,10.0,90.0,85.5,1026.0,4.5,85.5,False,True,False,True,monthly,True,NaT,False
24554,S024554,A06813,2026-02-27 12:42:17,2026-02-28 00:00:00,Enterprise,48,44.0,2112.0,1689.6,20275.2,422.4,1689.6,False,False,False,True,annual,True,NaT,False
191,S000191,A00052,2026-02-27 13:08:47,NaT,Free,21,0.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True,monthly,False,NaT,False
10377,S010377,A02870,2026-02-27 15:44:55,2026-02-28 00:00:00,Pro,63,25.0,1575.0,1417.5,17010.0,157.5,1417.5,False,False,False,True,monthly,True,NaT,False


In [ ]:
subscriptions[(subscriptions['account_id'] == 'A-82d8a6')].sort_values('start_date')

,subscription_id,account_id,start_date,end_date,plan_tier,seats,price,revenue,mrr_amount,arr_amount,discount_value,net_revenue,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,next_start,is_reactivation


In [ ]:
subscriptions[(subscriptions['upgrade_flag'] == True) & (subscriptions['upgrade_flag'] == subscriptions['downgrade_flag'])]

,subscription_id,account_id,start_date,end_date,plan_tier,seats,price,revenue,mrr_amount,arr_amount,discount_value,net_revenue,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,next_start,is_reactivation


In [ ]:
subscriptions[subscriptions['auto_renew_flag']==False].info()

<class 'pandas.DataFrame'>
Index: 14131 entries, 1 to 35819
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   subscription_id    14131 non-null  str           
 1   account_id         14131 non-null  str           
 2   start_date         14131 non-null  datetime64[us]
 3   end_date           11341 non-null  datetime64[us]
 4   plan_tier          14131 non-null  str           
 5   seats              14131 non-null  int64         
 6   price              14131 non-null  float64       
 7   revenue            14131 non-null  float64       
 8   mrr_amount         14131 non-null  float64       
 9   arr_amount         14131 non-null  float64       
 10  discount_value     14131 non-null  float64       
 11  net_revenue        14131 non-null  float64       
 12  is_trial           14131 non-null  bool          
 13  upgrade_flag       14131 non-null  bool          
 14  downgrade_flag     141

### Churn

In [ ]:
churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 3566 entries, 0 to 3565
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   churn_event_id            3566 non-null   str           
 1   account_id                3566 non-null   str           
 2   churn_date                2757 non-null   datetime64[us]
 3   reason_code               3566 non-null   str           
 4   refund_amount_usd         3566 non-null   float64       
 5   preceding_upgrade_flag    3566 non-null   bool          
 6   preceding_downgrade_flag  3566 non-null   bool          
 7   is_reactivation           3566 non-null   bool          
 8   feedback_text             2712 non-null   str           
dtypes: bool(3), datetime64[us](1), float64(1), str(4)
memory usage: 177.7 KB


In [ ]:
churn.sample()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
2636,C026596,A07388,2024-08-17 19:22:27,price_too_high,0.0,False,False,False,Not useful


In [ ]:
churn[churn['is_reactivation'] == True]

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
1,C000008,A00002,2024-12-21 13:12:42,competitor,0.0,False,True,True,Buggy
9,C000073,A00022,2025-07-04 13:24:49,no_longer_needed,0.0,False,False,True,Not useful
15,C000112,A00033,2025-02-20 14:59:44,missing_features,0.0,False,False,True,NaN
18,C000167,A00045,2025-11-26 02:58:07,bugs,0.0,False,False,True,Buggy
34,C000361,A00102,2025-10-22 07:55:32,price_too_high,0.0,True,False,True,Not useful
...,...,...,...,...,...,...,...,...,...
3502,C034941,A09763,2024-04-01 15:42:10,price_too_high,0.0,False,False,True,NaN
3506,C034989,A09774,2025-07-11 04:38:14,missing_features,0.0,True,False,True,NaN
3510,C035021,A09784,2025-10-16 01:58:37,competitor,0.0,False,False,True,Buggy
3534,C035420,A09892,2025-11-28 10:52:06,price_too_high,0.0,False,True,True,Too expensive


### Usage

In [ ]:
usage.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   usage_id             50000 non-null  str           
 1   subscription_id      50000 non-null  str           
 2   usage_date           50000 non-null  datetime64[us]
 3   feature_name         50000 non-null  str           
 4   usage_count          50000 non-null  int64         
 5   usage_duration_secs  50000 non-null  int64         
 6   error_count          50000 non-null  int64         
 7   is_beta_feature      50000 non-null  bool          
dtypes: bool(1), datetime64[us](1), int64(3), str(3)
memory usage: 2.7 MB


In [ ]:
tickets.sample()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
3458,T003458,A04232,2025-11-27 11:10:09,2025-11-30 10:22:09,71.2,low,77,NaN,False


In [ ]:
usage[usage['subscription_id'] == 'S00017']

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature


In [ ]:
usage.sample()

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
20941,U003321,S003321,2025-10-06 04:23:25,api,22,726,1,False


### Tickets

In [ ]:
tickets.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   ticket_id                    4000 non-null   str           
 1   account_id                   4000 non-null   str           
 2   submitted_at                 4000 non-null   datetime64[us]
 3   closed_at                    4000 non-null   datetime64[us]
 4   resolution_time_hours        4000 non-null   float64       
 5   priority                     4000 non-null   str           
 6   first_response_time_minutes  4000 non-null   int64         
 7   satisfaction_score           2429 non-null   float64       
 8   escalation_flag              4000 non-null   bool          
dtypes: bool(1), datetime64[us](2), float64(2), int64(1), str(3)
memory usage: 254.0 KB


In [ ]:
tickets.sample()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
1572,T001572,A00060,2025-10-06 08:30:13,2025-10-07 02:06:13,17.6,high,81,4.27,False
